# Observation-masked NPE — Case study III: *Xylella fastidiosa* with explicit vector dynamics

Applies OM-NPE to the host–vector model of

> Giménez-Romero À., Moralejo E. & Matías M.A. (2023) A compartmental model for
> *Xylella fastidiosa* diseases with explicit vector seasonal dynamics.
> *Phytopathology* **113**, 1686–1696.

using the **same model, priors and data** as their Bayesian (MCMC) fit to the olive
quick decline syndrome (OQDS) outbreak in Apulia, so that the amortised posterior
can be compared directly with their published posterior (their Table 2).

| Output | Content |
|---|---|
| `table_vector_model.csv` | Posterior summaries: OM-NPE vs Giménez-Romero *et al.* (2023), Table 2 |
| `figure_v1.pdf` | Prior and posterior marginals, joint posterior of $\alpha$ and $\beta$, and posterior simulations of hosts and vectors |
| `figure_v2.pdf` | Joint posterior of the transmission rates $\alpha$ and $\beta$ |
| `figure_v3.pdf` | Posterior simulations of the four host compartments |
| `figure_v4.pdf` | Posteriors under alternative survey schedules (masks) |
| `figure_v5.pdf` | Simulation-based calibration rank CDFs |

**Model.** Deterministic SEIR-V model (their eqn 2): hosts $S_H, E_H, I_H, R_H$ and
vectors $S_v, I_v$, with a yearly pulse of newly emerged susceptible vectors.
Six parameters are inferred, as in the original fit:
$\theta = (\tau_I, \tau_E, \beta, \alpha, \mu, \sigma)$, where $\sigma$ is the
standard deviation of the Gaussian observation model.

**Observation masking.** The aggregated record is placed on a common 7-year grid
(years 4–10 after introduction) with a binary mask, giving a 21-dimensional
network input `[y_1 m_1, ..., y_7 m_7, m_1, ..., m_7]`, where each $y$ holds the
two fitted proportions $((S_H+E_H)/N_H,\ (I_H+R_H)/N_H)$.

**Data.** `OQDS_data.csv` — aggregated grove data digitised from Fig. 4 of White
*et al.* (2020), as processed and fitted by Giménez-Romero *et al.* (their GitHub
repository). Optional: `OQDS_mcmc_reference.csv`, draws from an MCMC fit of the
same model, priors and data, shown alongside the NPE posterior when present.

**Runtime.** About one minute of simulation and roughly ten minutes of training
for $10^5$ simulations. Set `QUICK = True` for a fast smoke test (results will not match).

## 1. Imports and configuration

In [ ]:
import os
import pickle
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats as sstats
from sbi.utils import BoxUniform

try:
    from sbi.inference import NPE
except ImportError:                      # older sbi releases
    from sbi.inference import SNPE as NPE

warnings.filterwarnings("ignore")

QUICK = False   # True -> tiny training/diagnostic budgets for a smoke test

def data_path(name):
    # data files are looked for in data/ first, then next to the notebook
    return os.path.join("data", name) if os.path.exists(os.path.join("data", name)) else name

# --- model structure (as in the authors' fitting script) ------------------------
DAYS = 365.0          # days per year; rates are per day
N_H = 2959            # olive trees in the aggregated dataset
NV_PER_HOST = 0.5     # newly emerged vectors per host each year, N_v(0) = N_H / 2
E0 = 0.006            # initial exposed proportion (from White et al. 2020); I0 = 0, S0 = 1
N_YEARS = 16          # simulated years (fits use years 4-10)
DT = 1.0              # RK4 step in days (error < 1e-7 in the fitted proportions)
YEAR_GRID = np.arange(4, 11)                  # survey years on the common grid
INPUT_DIM = 2 * len(YEAR_GRID) + len(YEAR_GRID)

# --- inference budgets ---------------------------------------------------------
N_TRAIN = 20_000 if QUICK else 100_000
N_POST = 10_000                           # posterior draws for the observed record
N_PPC = 1_000                             # posterior-predictive trajectories
N_SBC = 200 if QUICK else 2_000           # SBC trials
L_SBC = 200                               # posterior draws per SBC trial
BATCH = 20_000                            # simulation batch size
MASK_MIX_P = 0.5                          # P(full 7-year schedule) vs P(random subset)

SEED = 0

# --- parameters and prior (authors' Turing.jl script for OQDS) -------------------
# tau_I ~ N(3.5, 1) on (0.01, 10)      host infectious period (years)
# tau_E ~ N(1.75, 0.5) on (0.1, 4)     host latent period (years)
# beta  ~ U(0.01, 10)                  vector-to-host transmission rate (per day)
# alpha ~ U(1e-5, 1e-2)                host-to-vector transmission rate (per day)
# mu    ~ N(0.02, 0.0075) on (0.01, 0.04)   vector removal rate (per day)
# sigma ~ InverseGamma(10, 1)          s.d. of the Gaussian observation model
PARAMS = ["tau_I", "tau_E", "beta", "alpha", "mu", "sigma"]
# BoxUniform support for the estimator: the prior bounds; for sigma a box holding
# all but ~1e-6 of the InverseGamma(10, 1) mass.
LOW = np.array([0.01, 0.1, 1e-2, 1e-5, 0.01, 0.03])
HIGH = np.array([10.0, 4.0, 10.0, 1e-2, 0.04, 0.60])

PARAM_LABELS = {"tau_I": r"$\tau_I$ (yr)", "tau_E": r"$\tau_E$ (yr)",
                "beta": r"$\beta$ (day$^{-1}$)", "alpha": r"$\alpha$ (day$^{-1}$)",
                "mu": r"$\mu$ (day$^{-1}$)", "sigma": r"$\sigma$", "R0": r"$R_0$"}

# Giménez-Romero et al. (2023), Table 2: posterior mean, median and interval.
# Their intervals are headed "95%", but correspond to the 5th-95th percentiles of
# the posterior (checked against an MCMC reproduction), so 5-95% is used throughout.
PAPER = {"tau_I": (3.61, 3.60, 2.06, 5.20), "tau_E": (1.24, 1.25, 0.70, 1.75),
         "beta": (3.44, 2.60, 0.55, 8.79), "alpha": (0.0031, 0.0022, 0.0005, 0.0084),
         "mu": (0.0240, 0.0240, 0.014, 0.035), "R0": (33.0, 21.0, np.nan, np.nan)}

COL_SE, COL_IR = "blue", "brown"
COL_NPE, COL_REF, COL_PRIOR = "#2a78d6", "#d95f02", "black"

torch.manual_seed(SEED)
print(f"Estimated parameters: {PARAMS}  | N_TRAIN = {N_TRAIN:,}  | INPUT_DIM = {INPUT_DIM}  | QUICK = {QUICK}")

## 2. Simulator: SEIR-V model with a yearly pulse of vectors

$$\dot S_H = -\beta S_H I_v/N_H,\quad \dot E_H = \beta S_H I_v/N_H - \kappa E_H,\quad
  \dot I_H = \kappa E_H - \gamma I_H,\quad \dot R_H = \gamma I_H,$$
$$\dot S_v = -\alpha S_v I_H/N_H - \mu S_v,\quad \dot I_v = \alpha S_v I_H/N_H - \mu I_v,$$

with $\kappa = 1/\tau_E$, $\gamma = 1/\tau_I$ (converted to per day), and $S_v$ reset
to $N_v(0)$ every 365 days. All six states are divided by $N_H$; the equations keep
the same form, so the model is integrated in per-host units. As in the original
fit, the observation model is Gaussian with standard deviation $\sigma$ on the two
proportions $(S_H+E_H)/N_H$ and $(I_H+R_H)/N_H$.

In [ ]:
def seirv_batch(theta, n_years=N_YEARS, dt=DT, daily_vectors=False):
    '''Vectorised SEIR-V simulator (fixed-step RK4, as in the authors' code).

    Parameters
    ----------
    theta : (bs, >=5) array
        Columns tau_I, tau_E, beta, alpha, mu (any further columns are ignored).

    Returns
    -------
    dict of (bs, n_years + 1) arrays
        Host proportions S, E, I, R and the yearly peak of I_v / N_v(0), at years 0..n_years.
        With daily_vectors=True also 'Iv_daily', I_v / N_v(0) after every integration step.
    '''
    tau_I, tau_E, beta, alpha, mu = [np.asarray(theta[:, i], float) for i in range(5)]
    kappa, gamma = 1.0 / (tau_E * DAYS), 1.0 / (tau_I * DAYS)
    bs = theta.shape[0]
    S, E, I, R = np.ones(bs), np.full(bs, E0), np.zeros(bs), np.zeros(bs)
    Sv, Iv = np.full(bs, NV_PER_HOST), np.zeros(bs)
    out = {k: np.empty((bs, n_years + 1)) for k in ("S", "E", "I", "R", "Iv_peak")}
    out["S"][:, 0], out["E"][:, 0], out["I"][:, 0], out["R"][:, 0], out["Iv_peak"][:, 0] = S, E, I, R, 0.0

    def rhs(S, E, I, Sv, Iv):
        inf_h = beta * S * Iv
        inf_v = alpha * Sv * I
        return -inf_h, inf_h - kappa * E, kappa * E - gamma * I, gamma * I, -inf_v - mu * Sv, inf_v - mu * Iv

    n_steps = int(round(DAYS / dt))
    if daily_vectors:
        out["Iv_daily"] = np.zeros((bs, n_years * n_steps + 1))
    for year in range(1, n_years + 1):
        iv_peak = np.zeros(bs)
        for step in range(n_steps):
            k1 = rhs(S, E, I, Sv, Iv)
            k2 = rhs(S + 0.5 * dt * k1[0], E + 0.5 * dt * k1[1], I + 0.5 * dt * k1[2],
                     Sv + 0.5 * dt * k1[4], Iv + 0.5 * dt * k1[5])
            k3 = rhs(S + 0.5 * dt * k2[0], E + 0.5 * dt * k2[1], I + 0.5 * dt * k2[2],
                     Sv + 0.5 * dt * k2[4], Iv + 0.5 * dt * k2[5])
            k4 = rhs(S + dt * k3[0], E + dt * k3[1], I + dt * k3[2], Sv + dt * k3[4], Iv + dt * k3[5])
            S = S + dt / 6 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
            E = E + dt / 6 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
            I = I + dt / 6 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
            R = R + dt / 6 * (k1[3] + 2 * k2[3] + 2 * k3[3] + k4[3])
            Sv = Sv + dt / 6 * (k1[4] + 2 * k2[4] + 2 * k3[4] + k4[4])
            Iv = Iv + dt / 6 * (k1[5] + 2 * k2[5] + 2 * k3[5] + k4[5])
            iv_peak = np.maximum(iv_peak, Iv)
            if daily_vectors:
                out["Iv_daily"][:, (year - 1) * n_steps + step + 1] = Iv / NV_PER_HOST
        Sv = np.full(bs, NV_PER_HOST)            # yearly pulse: S_v reset to N_v(0)
        out["S"][:, year], out["E"][:, year], out["I"][:, year], out["R"][:, year] = S, E, I, R
        out["Iv_peak"][:, year] = iv_peak / NV_PER_HOST
    return out


def fitted_proportions(sim):
    '''(bs, n_years + 1, 2): the two quantities fitted to data, (S+E)/N and (I+R)/N.'''
    return np.stack([sim["S"] + sim["E"], sim["I"] + sim["R"]], axis=2)


def R0_fun(theta, s=1.0, tau=DAYS):
    '''Approximate basic reproductive number (their eqn 3), one generation = one year.'''
    tau_I, beta, alpha, mu = theta[:, 0], theta[:, 2], theta[:, 3], theta[:, 4]
    gamma = 1.0 / (tau_I * DAYS)
    return beta * alpha / (gamma * mu) * s * NV_PER_HOST / (mu * tau) * (1 - np.exp(-mu * tau))


# check: published median parameters reproduce the fitted curve of their Fig. 4D
_chk = fitted_proportions(seirv_batch(np.array([[3.60, 1.25, 2.60, 0.0022, 0.0240]])))[0]
print("S+E at years 4-10 for the published medians:", np.round(_chk[YEAR_GRID, 0], 3))

## 3. Survey data

Aggregated proportions for 17 olive groves in Apulia, digitised from Fig. 4 of
White *et al.* (2020). Following Giménez-Romero *et al.*, symptomatic plus
desiccated trees ($I_S + I_D$) are fitted by $I_H + R_H$, and susceptible plus
symptomless trees ($S + I_A$) by $S_H + E_H$, for years 4–10.

In [ ]:
oqds = pd.read_csv(data_path("OQDS_data.csv"), index_col=0)
oqds = oqds[oqds["Years"].isin(YEAR_GRID)].reset_index(drop=True)
oqds["I_R"] = oqds["I"] + oqds["R"]
OBS_YEARS = oqds["Years"].to_numpy(int)
OBS = oqds[["S_E", "I_R"]].to_numpy(float)            # (7, 2)

# error bars as in the authors' processing: s.d. across groves, clipped to [0, 1]
ir_err = (oqds["I_err"] + oqds["R_err"]).to_numpy()
ERR = {"S_E": (np.minimum(oqds["S_E_err"], oqds["S_E"]).to_numpy(), oqds["S_E_err"].to_numpy()),
       "I_R": (ir_err, np.minimum(ir_err, 1 - oqds["I_R"]).to_numpy())}
print(oqds[["Years", "S_E", "I_R"]].round(3).to_string(index=False))

## 4. Observation-masked input

In [ ]:
YEAR_IDX = {int(y): i for i, y in enumerate(YEAR_GRID)}

def build_input(prop_rows, years_obs):
    '''Masked network input: [proportions * mask, mask].'''
    x = np.zeros(2 * len(YEAR_GRID))
    m = np.zeros(len(YEAR_GRID))
    for yr, p in zip(years_obs, prop_rows):
        i = YEAR_IDX[int(yr)]
        x[2 * i:2 * i + 2] = p
        m[i] = 1.0
    return np.concatenate([x, m])


assert len(build_input(OBS, OBS_YEARS)) == INPUT_DIM
print("NPE input dimension:", INPUT_DIM)

## 5. Prior, training simulations and estimator

Parameters are drawn from the prior of the original fit. Each simulated record is
the model trajectory at the survey years plus Gaussian noise with the drawn
$\sigma$. Observation schedules are a mixture: with probability 0.5 all seven
survey years are observed (the schedule of the real record), otherwise a random
subset of two to seven years. Masks are generated independently of the
parameters.

In [ ]:
def sample_prior(n, rng):
    '''Draw n parameter vectors: tau_I, tau_E, beta, alpha, mu, sigma.'''
    def tnorm(m, s, a, b):
        return sstats.truncnorm.rvs((a - m) / s, (b - m) / s, loc=m, scale=s, size=n,
                                    random_state=int(rng.integers(2 ** 31)))
    sigma = np.empty(n); filled = 0
    while filled < n:                       # InverseGamma(10, 1) restricted to the box
        s = 1.0 / rng.gamma(10.0, 1.0, size=n)
        s = s[(s > LOW[5]) & (s < HIGH[5])][:n - filled]
        sigma[filled:filled + len(s)] = s; filled += len(s)
    return np.column_stack([tnorm(3.5, 1.0, 0.01, 10.0), tnorm(1.75, 0.5, 0.1, 4.0),
                            rng.uniform(0.01, 10.0, n), rng.uniform(1e-5, 1e-2, n),
                            tnorm(0.02, 0.0075, 0.01, 0.04), sigma])


def prior_pdf(p, x):
    '''Prior density of one parameter (for plotting).'''
    tn = lambda m, s, a, b: sstats.truncnorm.pdf(x, (a - m) / s, (b - m) / s, loc=m, scale=s)
    return {"tau_I": lambda: tn(3.5, 1.0, 0.01, 10.0), "tau_E": lambda: tn(1.75, 0.5, 0.1, 4.0),
            "beta": lambda: sstats.uniform.pdf(x, 0.01, 9.99), "alpha": lambda: sstats.uniform.pdf(x, 1e-5, 1e-2 - 1e-5),
            "mu": lambda: tn(0.02, 0.0075, 0.01, 0.04), "sigma": lambda: sstats.invgamma.pdf(x, 10.0, scale=1.0)}[p]()


def sample_mask(rng):
    '''Draw one training example's observed-years schedule.'''
    if rng.random() < MASK_MIX_P:
        return YEAR_GRID.copy()
    k = rng.integers(2, len(YEAR_GRID) + 1)
    return np.sort(rng.choice(YEAR_GRID, size=k, replace=False))


def simulate_records(theta, rng):
    '''Noisy fitted proportions at the grid years, shape (bs, 7, 2).'''
    prop = fitted_proportions(seirv_batch(theta, n_years=int(YEAR_GRID.max())))[:, YEAR_GRID, :]
    return prop + theta[:, 5][:, None, None] * rng.standard_normal(prop.shape)


NPE_PATH = f"xfv_npe_posterior_N{N_TRAIN}.pkl"
prior = BoxUniform(low=torch.tensor(LOW, dtype=torch.float32),
                   high=torch.tensor(HIGH, dtype=torch.float32))

if os.path.exists(NPE_PATH):
    print(f"Loading cached estimator from {NPE_PATH}")
    with open(NPE_PATH, "rb") as f:
        posterior = pickle.load(f)
else:
    rng_train = np.random.default_rng(SEED)
    theta_train = sample_prior(N_TRAIN, rng_train)
    X = np.empty((N_TRAIN, INPUT_DIM))

    t0 = time.perf_counter()
    for start in range(0, N_TRAIN, BATCH):
        end = min(start + BATCH, N_TRAIN)
        rec = simulate_records(theta_train[start:end], rng_train)
        for i in range(start, end):
            yrs_i = sample_mask(rng_train)
            X[i] = build_input(rec[i - start, yrs_i - YEAR_GRID[0]], yrs_i)
        if end % 100_000 == 0 or end == N_TRAIN:
            rate = end / (time.perf_counter() - t0)
            print(f"  simulated {end:,}/{N_TRAIN:,}  (ETA {(N_TRAIN - end) / rate / 60:.1f} min)")
    print(f"Simulation completed in {(time.perf_counter() - t0) / 60:.1f} min")

    t1 = time.perf_counter()
    inference = NPE(prior=prior)
    inference.append_simulations(torch.tensor(theta_train, dtype=torch.float32),
                                torch.tensor(X, dtype=torch.float32)).train()
    posterior = inference.build_posterior()
    print(f"Training completed in {(time.perf_counter() - t1) / 60:.1f} min")

    with open(NPE_PATH, "wb") as f:
        pickle.dump(posterior, f)
    print(f"Saved estimator to {NPE_PATH}")

## 6. Posterior for the aggregated OQDS record

One forward pass of the trained estimator, conditioned on the seven observed years.

In [ ]:
def npe_sample(prop_rows, years_obs, n):
    x = torch.tensor(build_input(prop_rows, years_obs), dtype=torch.float32)
    return posterior.sample((n,), x=x, show_progress_bars=False).numpy().astype(float)


t_infer = time.perf_counter()
samples = npe_sample(OBS, OBS_YEARS, N_POST)
print(f"Posterior evaluation ({N_POST:,} draws): {time.perf_counter() - t_infer:.2f}s")
R0_samples = R0_fun(samples)

# optional reference: MCMC draws for the same model, priors and data
REF_PATH = data_path("OQDS_mcmc_reference.csv")
ref = pd.read_csv(REF_PATH)[PARAMS].to_numpy(float) if os.path.exists(REF_PATH) else None
print("MCMC reference draws:", "none found" if ref is None else f"{len(ref):,} from {REF_PATH}")

## 7. Table — OM-NPE compared with Giménez-Romero *et al.* (2023)

$R_0$ is computed for every posterior draw with their eqn 3 and then summarised.

In [ ]:
def summarise(v):
    return np.mean(v), np.median(v), np.percentile(v, 5), np.percentile(v, 95)

rows = []
for j, p in enumerate(PARAMS[:5] + ["R0", "sigma"]):
    v = R0_samples if p == "R0" else samples[:, PARAMS.index(p)]
    mean, med, lo, hi = summarise(v)
    row = {"parameter": p, "npe_mean": mean, "npe_median": med, "npe_q05": lo, "npe_q95": hi}
    if p in PAPER:
        row.update(dict(zip(["paper_mean", "paper_median", "paper_lo", "paper_hi"], PAPER[p])))
    if ref is not None:
        r = R0_fun(ref) if p == "R0" else ref[:, PARAMS.index(p)]
        row.update(dict(zip(["mcmc_mean", "mcmc_median", "mcmc_q05", "mcmc_q95"], summarise(r))))
    rows.append(row)

table = pd.DataFrame(rows)
with pd.option_context("display.float_format", lambda v: f"{v:.3g}", "display.width", 200):
    print(table.to_string(index=False))
table.to_csv("table_vector_model.csv", index=False)

lab = np.log(samples[:, PARAMS.index("alpha")]), np.log(samples[:, PARAMS.index("beta")])
print(f"\nPosterior correlation of log(alpha) and log(beta): {np.corrcoef(*lab)[0, 1]:+.2f}")

## 8. Posterior simulations

The model is simulated for draws from the OM-NPE posterior and, when present, from
the MCMC reference. Trajectories are summarised by their median and 99% interval,
as in Fig. 4D of the original paper.

In [ ]:
rng_ppc = np.random.default_rng(1)
th_ppc = samples[rng_ppc.choice(len(samples), N_PPC, replace=False)]
sim_ppc = seirv_batch(th_ppc, daily_vectors=True)
prop_ppc = fitted_proportions(sim_ppc)                 # (N_PPC, 17, 2)
sim_ref = prop_ref = None
if ref is not None:
    sim_ref = seirv_batch(ref[rng_ppc.choice(len(ref), N_PPC, replace=False)], daily_vectors=True)
    prop_ref = fitted_proportions(sim_ref)
grid = np.arange(N_YEARS + 1)
t_daily = np.arange(sim_ppc["Iv_daily"].shape[1]) * DT / DAYS

rmse = lambda prop: np.sqrt(np.mean((np.median(prop, axis=0)[OBS_YEARS] - OBS) ** 2))
print(f"RMSE of the posterior-median trajectory to the data: OM-NPE {rmse(prop_ppc):.3f}"
      + ("" if prop_ref is None else f", MCMC reference {rmse(prop_ref):.3f}"))
if prop_ref is not None:
    gap = np.abs(np.median(prop_ppc, axis=0) - np.median(prop_ref, axis=0)).max()
    print(f"Largest difference between the two median host trajectories (years 0-{N_YEARS}): {gap:.3f}")
def peak_summary(sim):
    lo, med, hi = np.percentile(sim["Iv_peak"].max(axis=1), [5, 50, 95])
    return f"{med:.3f} ({lo:.3f}-{hi:.3f})"
print("Largest yearly peak of I_v/N_v(0), median (5-95% interval): OM-NPE " + peak_summary(sim_ppc)
      + ("" if sim_ref is None else ", MCMC reference " + peak_summary(sim_ref)))

## 9. Figure V1 — parameter distributions, joint posterior and posterior simulations

Prior and posterior marginals (a-f), the joint posterior of the two transmission
rates for each method (g, h), and simulated host proportions with the data (i) and
the simulated proportion of infected vectors (j) for both methods.

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 10,
                     "xtick.labelsize": 8, "ytick.labelsize": 8})
LOG_AXES = {"alpha", "beta"}
XLIM = {"tau_I": (0, 10), "tau_E": (0, 4), "mu": (0.005, 0.04), "alpha": (1e-5, 1e-2),
        "beta": (1e-2, 10), "sigma": (0.0, 0.25)}

fig = plt.figure(figsize=(19.5, 6.6))
gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 1.25, 1.5], wspace=0.38, hspace=0.38)

# (i) host proportions fitted to data
ax = fig.add_subplot(gs[0, 4])
for k, (lab_k, col, key) in enumerate([("S + E", COL_SE, "S_E"), ("I + R", COL_IR, "I_R")]):
    lo, med, hi = np.percentile(prop_ppc[:, :, k], [0.5, 50, 99.5], axis=0)
    ax.fill_between(grid, lo, hi, color="0.85", lw=0)
    ax.plot(grid, med, color=col, lw=2, label=f"{lab_k}, OM-NPE")
    if prop_ref is not None:
        lo_r, med_r, hi_r = np.percentile(prop_ref[:, :, k], [0.5, 50, 99.5], axis=0)
        ax.plot(grid, med_r, color=col, lw=1.6, ls="--", label=f"{lab_k}, MCMC")
        ax.plot(grid, lo_r, color=col, lw=0.8, ls=":")
        ax.plot(grid, hi_r, color=col, lw=0.8, ls=":")
    ax.errorbar(OBS_YEARS, OBS[:, k], yerr=[np.nan_to_num(ERR[key][0]), np.nan_to_num(ERR[key][1])],
                fmt="o", color=col, ms=4.5, capsize=3, lw=1.2)
ax.fill_between([], [], [], color="0.85", label="99% interval, OM-NPE")
if prop_ref is not None:
    ax.plot([], [], color="0.3", lw=0.8, ls=":", label="99% interval, MCMC")
ax.set_xlabel("Time [years]"); ax.set_ylabel("Proportion of hosts"); ax.set_ylim(-0.02, 1.05)
ax.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.42), fontsize=7)
ax.set_title("i", loc="left", fontweight="bold")

# (j) infected vectors
ax = fig.add_subplot(gs[1, 4])
lo, med, hi = np.percentile(sim_ppc["Iv_daily"], [2.5, 50, 97.5], axis=0)
ax.fill_between(t_daily, lo, hi, color=COL_NPE, alpha=0.25, lw=0, label="95% interval, OM-NPE")
ax.plot(t_daily, med, color=COL_NPE, lw=1.2, label="Median, OM-NPE")
if sim_ref is not None:
    lo_r, med_r, hi_r = np.percentile(sim_ref["Iv_daily"], [2.5, 50, 97.5], axis=0)
    ax.plot(t_daily, med_r, color=COL_REF, lw=1.0, ls="--", label="Median, MCMC")
    ax.plot(t_daily, hi_r, color=COL_REF, lw=0.7, ls=":", label="95% interval, MCMC")
ax.set_xlabel("Time [years]"); ax.set_ylabel(r"Infected vectors, $I_v/N_v(0)$")
ax.set_xlim(0, N_YEARS); ax.set_ylim(bottom=0)
ax.legend(frameon=False, loc="upper right", fontsize=7)
ax.set_title("j", loc="left", fontweight="bold")

# (a-f) prior and posterior marginals
for n, p in enumerate(["tau_I", "tau_E", "mu", "alpha", "beta", "sigma"]):
    ax = fig.add_subplot(gs[n // 3, n % 3])
    j = PARAMS.index(p)
    lo, hi = XLIM[p]
    bins = np.geomspace(lo, hi, 60) if p in LOG_AXES else np.linspace(lo, hi, 60)
    ax.hist(samples[:, j], bins=bins, density=True, color=COL_NPE, alpha=0.75, label="OM-NPE posterior")
    if ref is not None:
        ax.hist(ref[:, j], bins=bins, density=True, histtype="step", color=COL_REF, lw=1.4,
                label="MCMC posterior")
    xs = np.geomspace(max(lo, 1e-6), hi, 400) if p in LOG_AXES else np.linspace(max(lo, 1e-3), hi, 400)
    ax.plot(xs, prior_pdf(p, xs), color=COL_PRIOR, lw=1.8, label="Prior")
    if p in LOG_AXES:
        ax.set_xscale("log")
    ax.set_xlim(lo, hi)
    ax.set_xlabel(PARAM_LABELS[p]); ax.set_ylabel("PDF")
    ax.set_title("abcdef"[n], loc="left", fontweight="bold")
    if n == 0:
        h, l = ax.get_legend_handles_labels()

# (g, h) joint posterior of the two transmission rates
ja, jb = PARAMS.index("alpha"), PARAMS.index("beta")
joint = [("OM-NPE", samples)] + ([("MCMC", ref)] if ref is not None else [])
for r, (name, smp) in enumerate(joint):
    ax = fig.add_subplot(gs[r, 3] if len(joint) == 2 else gs[:, 3])
    hb = ax.hexbin(smp[:, ja], smp[:, jb], gridsize=40, xscale="log", yscale="log",
                   cmap="viridis_r", mincnt=1, linewidths=0, extent=(-4, -2, -1, 1))
    ax.set_xlim(1e-4, 1e-2); ax.set_ylim(1e-1, 10)
    ax.set_xlabel(PARAM_LABELS["alpha"]); ax.set_ylabel(PARAM_LABELS["beta"])
    ax.text(0.05, 0.06, name, transform=ax.transAxes, ha="left", va="bottom", fontsize=9)
    cb = fig.colorbar(hb, ax=ax, pad=0.02); cb.set_label("draws per cell", fontsize=7)
    cb.ax.tick_params(labelsize=6)
    ax.set_title("gh"[r], loc="left", fontweight="bold")

for ax in fig.axes:
    if ax.get_label() != "<colorbar>":
        ax.spines[["top", "right"]].set_visible(False)
fig.legend(h, l, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.30, 0.97))
fig.savefig("figure_v1.pdf", bbox_inches="tight")
fig.savefig("figure_v1.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Figure V2 — joint posterior of the transmission rates

Without data on the vector population, many combinations of $\alpha$ and $\beta$
fit the host data equally well (the identifiability problem discussed by
Giménez-Romero *et al.*). The joint posterior shows this as a ridge.

In [ ]:
ja, jb = PARAMS.index("alpha"), PARAMS.index("beta")
fig, axes = plt.subplots(1, 2 if ref is not None else 1, figsize=(5.2 * (2 if ref is not None else 1), 4.3),
                         sharex=True, sharey=True, squeeze=False)
for ax, (name, s) in zip(axes[0], [("OM-NPE", samples)] + ([("MCMC posterior", ref)] if ref is not None else [])):
    hb = ax.hexbin(s[:, ja], s[:, jb], gridsize=45, xscale="log", yscale="log",
                   cmap="viridis_r", mincnt=1, linewidths=0)
    ax.plot(PAPER["alpha"][1], PAPER["beta"][1], "*", ms=14, color="red", mec="k", label="Published medians")
    ax.set_xlim(1e-4, 1e-2); ax.set_ylim(1e-1, 10)
    ax.set_xlabel(PARAM_LABELS["alpha"]); ax.set_title(name)
    ax.spines[["top", "right"]].set_visible(False)
    fig.colorbar(hb, ax=ax, pad=0.02).set_label("posterior draws per cell")
axes[0, 0].set_ylabel(PARAM_LABELS["beta"])
axes[0, 0].legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig("figure_v2.pdf", bbox_inches="tight")
fig.savefig("figure_v2.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Figure V3 — host compartments

Simulated proportions in the four host compartments for the OM-NPE posterior
(median and 5-95% interval).

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
for key, col, lab_k in [("S", "forestgreen", r"$S_H$"), ("E", "orange", r"$E_H$"),
                        ("I", "red", r"$I_H$"), ("R", "black", r"$R_H$")]:
    lo, med, hi = np.percentile(sim_ppc[key], [5, 50, 95], axis=0)
    ax.fill_between(grid, lo, hi, color=col, alpha=0.15, lw=0)
    ax.plot(grid, med, color=col, lw=2, label=lab_k)
ax.set_xlabel("Time [years]"); ax.set_ylabel("Proportion of hosts"); ax.set_ylim(-0.02, 1.05)
ax.legend(frameon=False, ncol=4, loc="upper center"); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("figure_v3.pdf", bbox_inches="tight")
fig.savefig("figure_v3.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. Figure V4 — posteriors under alternative survey schedules

The same trained estimator is evaluated on subsets of the survey years, without
retraining: the full record, early years only, late years only, and three
spread-out years.

In [ ]:
SCHEDULES = {"all years (4-10)": [4, 5, 6, 7, 8, 9, 10], "early (4-6)": [4, 5, 6],
             "late (8-10)": [8, 9, 10], "spread (4, 7, 10)": [4, 7, 10]}
sched_samples = {}
for name, yrs in SCHEDULES.items():
    idx = [int(np.where(OBS_YEARS == y)[0][0]) for y in yrs]
    s = npe_sample(OBS[idx], yrs, N_POST)
    sched_samples[name] = np.column_stack([s, R0_fun(s)])

shown = ["tau_I", "tau_E", "beta", "alpha", "mu", "R0"]
fig, axes = plt.subplots(2, 3, figsize=(10.5, 5.6))
for ax, p in zip(axes.ravel(), shown):
    j = len(PARAMS) if p == "R0" else PARAMS.index(p)
    for c, (name, s) in enumerate(sched_samples.items()):
        lo, med, hi = np.percentile(s[:, j], [5, 50, 95])
        ax.errorbar(c, med, yerr=[[med - lo], [hi - med]], fmt="o", color=COL_NPE, capsize=3, lw=1.4)
    if p in PAPER:
        ax.axhline(PAPER[p][1], color="k", ls="--", lw=1.0, label="Published median")
    if p in ("alpha", "beta", "R0"):
        ax.set_yscale("log")
    ax.set_xticks(range(len(SCHEDULES))); ax.set_xticklabels(list(SCHEDULES), rotation=25, ha="right")
    ax.set_ylabel(PARAM_LABELS[p]); ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("OM-NPE posterior median and 5-95% interval by survey schedule", fontsize=10)
fig.tight_layout()
fig.savefig("figure_v4.pdf", bbox_inches="tight")
fig.savefig("figure_v4.png", dpi=300, bbox_inches="tight")
plt.show()

## 13. Simulation-based calibration

Draw $\theta_k$ from the prior, simulate a record observed in **all seven survey
years** (the schedule of the real record), and rank the true $\theta_k$ among $L$
posterior draws. Under calibration the ranks are uniform.

In [ ]:
rng_sbc = np.random.default_rng(2027)
theta_sbc = sample_prior(N_SBC, rng_sbc)
rec_sbc = simulate_records(theta_sbc, rng_sbc)

sbc_ranks = np.zeros((N_SBC, len(PARAMS)), dtype=int)
t_sbc = time.perf_counter()
for i in range(N_SBC):
    s = npe_sample(rec_sbc[i], YEAR_GRID, L_SBC)
    sbc_ranks[i] = (s < theta_sbc[i]).sum(axis=0)
pd.DataFrame(sbc_ranks, columns=[f"rank_{p}" for p in PARAMS]).to_csv("xfv_sbc_ranks.csv", index=False)
print(f"{N_SBC:,} SBC trials in {time.perf_counter() - t_sbc:.1f}s")

r_grid = np.arange(L_SBC + 1)
p_null = (r_grid + 1) / (L_SBC + 1)
ci_lo, ci_hi = sstats.binom.interval(0.95, N_SBC, p_null)
ci_lo, ci_hi = ci_lo / N_SBC, ci_hi / N_SBC

fig, axes = plt.subplots(2, 3, figsize=(10.2, 6.0), sharex=True, sharey=True)
for ax, p in zip(axes.ravel(), PARAMS):
    ranks_sorted = np.sort(sbc_ranks[:, PARAMS.index(p)])
    ecdf = np.searchsorted(ranks_sorted, r_grid, side="right") / N_SBC
    ax.fill_between(r_grid / L_SBC, ci_lo, ci_hi, color="0.5", alpha=0.3, zorder=0)
    ax.plot([0, 1], [0, 1], color="k", lw=1, ls="--", zorder=1)
    ax.plot(r_grid / L_SBC, ecdf, color="tab:blue", lw=2, zorder=2)
    ax.set_title(PARAM_LABELS[p]); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Normalised rank"); ax.set_ylabel("Empirical CDF")
fig.tight_layout()
fig.savefig("figure_v5.pdf", bbox_inches="tight")
fig.savefig("figure_v5.png", dpi=300, bbox_inches="tight")
plt.show()

for q in (0.5, 0.9):
    lo_r, hi_r = (1 - q) / 2 * L_SBC, (1 + q) / 2 * L_SBC
    cov = ((sbc_ranks >= lo_r) & (sbc_ranks <= hi_r)).mean(axis=0)
    print(f"empirical coverage of the central {int(100 * q)}% interval:",
          dict(zip(PARAMS, np.round(cov, 3))))